# Cassava Leaf Disease Classification - MobileNetV2

## **1. Imports & Configuration**

In [76]:
# Operating system utilities
import os

# Data manipulation
import numpy as np
import pandas as pd

# Image processing and visualization
import cv2
import matplotlib.pyplot as plt

# Deep Learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.utils.class_weight import compute_class_weight
# Reproducibility
import random

In [77]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [78]:
# Dataset configuration
BASE_DIR = "/kaggle/input/notebooks/rawanhossamsaid/dl-preprocessing"
IMAGE_DIR = os.path.join(BASE_DIR, "processed")

# Image configuration
IMG_SIZE = 256
CHANNELS = 3

# Model configuration
NUM_CLASSES = 5
BATCH_SIZE = 16
SEED = 42

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))
print("Image directory:", IMAGE_DIR)

TensorFlow version: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Image directory: /kaggle/input/notebooks/rawanhossamsaid/dl-preprocessing/processed


# 2. Load and inspect the prepared dataset

In [79]:
# Load the training dataset
train_df = pd.read_csv(
    os.path.join(BASE_DIR, "train_split.csv")
)

# Load the validation dataset
val_df = pd.read_csv(
    os.path.join(BASE_DIR, "val_split.csv")
)

# Load the testing dataset
test_df = pd.read_csv(
    os.path.join(BASE_DIR, "test_split.csv")
)

print("Dataset loaded successfully!")

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

Dataset loaded successfully!
Training samples: 12838
Validation samples: 4279
Testing samples: 4280


In [80]:
# Display dataset information
for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{'='*40}")
    print(f"{name} Dataset")
    print(f"{'='*40}")

    print("Shape:", df.shape)
    print("\nColumns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())

    print("\nClass distribution:")
    print(df["label"].value_counts().sort_index())


Training Dataset
Shape: (12838, 4)

Columns: ['image_id', 'label', 'path', 'proc_path']

First 5 rows:


,image_id,label,path,proc_path
0,2571585254.jpg,0,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/2571585254.png
1,3034455648.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/3034455648.png
2,1630304159.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/1630304159.png
3,1454850686.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/1454850686.png
4,314571424.jpg,1,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/314571424.png



Class distribution:
label
0     652
1    1313
2    1432
3    7895
4    1546
Name: count, dtype: int64

Validation Dataset
Shape: (4279, 4)

Columns: ['image_id', 'label', 'path', 'proc_path']

First 5 rows:


,image_id,label,path,proc_path
0,2179057801.jpg,4,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/2179057801.png
1,1771017079.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/1771017079.png
2,1024067372.jpg,4,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/1024067372.png
3,2822765524.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/2822765524.png
4,496924131.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/496924131.png



Class distribution:
label
0     218
1     438
2     477
3    2631
4     515
Name: count, dtype: int64

Testing Dataset
Shape: (4280, 4)

Columns: ['image_id', 'label', 'path', 'proc_path']

First 5 rows:


,image_id,label,path,proc_path
0,1688478980.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/1688478980.png
1,632236115.jpg,1,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/632236115.png
2,487710367.jpg,4,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/487710367.png
3,2730357599.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/2730357599.png
4,3530853460.jpg,3,/kaggle/input/competitions/cassava-leaf-diseas...,/kaggle/working/processed/3530853460.png



Class distribution:
label
0     217
1     438
2     477
3    2632
4     516
Name: count, dtype: int64


# 3. Resolve and verify processed image paths

In [81]:
# Update processed image paths to the new Kaggle input directory

for df in [train_df, val_df, test_df]:
    df["proc_path"] = df["image_id"].apply(
        lambda name: os.path.join(
            IMAGE_DIR,
            os.path.splitext(name)[0] + ".png"
        )
    )

print("Image paths updated successfully!")

Image paths updated successfully!


In [82]:
# Check whether the first 5 processed images exist in each split

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")

    for path in df["proc_path"].head():
        print(os.path.basename(path), os.path.exists(path))


Train:
2571585254.png True
3034455648.png True
1630304159.png True
1454850686.png True
314571424.png True

Validation:
2179057801.png True
1771017079.png True
1024067372.png True
2822765524.png True
496924131.png True

Test:
1688478980.png True
632236115.png True
487710367.png True
2730357599.png True
3530853460.png True


In [83]:
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    sample = df["proc_path"].sample(
        min(100, len(df)),
        random_state=SEED
    )

    missing = sum(not os.path.exists(path) for path in sample)

    print(f"{name}: {missing} missing out of {len(sample)}")

Train: 0 missing out of 100
Validation: 0 missing out of 100
Test: 0 missing out of 100


# 4. Image loading and preprocessing

In [84]:
def load_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image

In [85]:
sample_image = load_image(train_df["proc_path"].iloc[0])

print("Image shape:", sample_image.shape)
print("Minimum value:", tf.reduce_min(sample_image).numpy())
print("Maximum value:", tf.reduce_max(sample_image).numpy())

Image shape: (256, 256, 3)
Minimum value: -1.0
Maximum value: 1.0


# 5. Build the data pipelines

In [86]:
def create_dataset(df, training=False):
    image_paths = df["proc_path"].values
    labels = df["label"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=SEED
        )

    dataset = dataset.map(
        lambda path, label: (load_image(path), label),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [87]:
train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("Data pipelines created successfully!")

Data pipelines created successfully!


# 6. MobileNetV2 backbone

In [88]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("MobileNetV2 loaded successfully!")
print("Total layers:", len(base_model.layers))
print("Trainable layers:", sum(layer.trainable for layer in base_model.layers))

/tmp/ipykernel_50/2450662679.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


MobileNetV2 loaded successfully!
Total layers: 154
Trainable layers: 0


# 7. Classification head and complete model

In [89]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation="relu")(x)

x = layers.Dropout(0.5)(x)

x = layers.Dense(64, activation="relu")(x)

x = layers.BatchNormalization()(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [90]:
## 9. Model Summary
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,435,909 (9.29 MB)

 Trainable params: 175,237 (684.52 KB)

 Non-trainable params: 2,260,672 (8.62 MB)

# 8. Compile the model

In [91]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


# 9. Architecture sanity checks

In [92]:
print("Model output shape:", model.output_shape)
print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Testing batches:", tf.data.experimental.cardinality(test_ds).numpy())

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Label values:", tf.unique(labels).y.numpy())

    probabilities = model(images, training=False)
    print("Forward-pass output shape:", probabilities.shape)
    print("First sample probability sum:", float(tf.reduce_sum(probabilities[0])))
    assert probabilities.shape == (BATCH_SIZE, NUM_CLASSES)

Model output shape: (None, 5)
Training batches: 803
Validation batches: 268
Testing batches: 268
Image batch shape: (16, 256, 256, 3)
Label batch shape: (16,)
Label values: [3 0 4 2]
Forward-pass output shape: (16, 5)
First sample probability sum: 1.0


In [93]:
ARCHITECTURE_PATH = "/kaggle/working/cassava_mobilenetv2.keras"
model.save(ARCHITECTURE_PATH)
print("Architecture saved to:", ARCHITECTURE_PATH)

Architecture saved to: /kaggle/working/cassava_mobilenetv2.keras


Handoff notes for Member 4
- Backbone: MobileNetV2 pretrained on ImageNet, with include_top=False.
- Input: 256 × 256 × 3 RGB images.
- Preprocessing: resize to 256 × 256, convert to float32, then apply MobileNetV2 preprocess_input. Do not divide by 255 again.
- Classifier head: GlobalAveragePooling2D → BatchNormalization → Dense(128, ReLU) → Dropout(0.5) → Dense(64, ReLU) → BatchNormalization → Dense(5, Softmax).
- Labels: integer labels 0–4 from the supplied split CSV files. Preserve the exact label-to-disease mapping documented by the preprocessing/data owner; do not infer or reorder it.
- Datasets: train_ds, val_ds, and test_ds are created here. Use training and validation for model development; reserve test for one final evaluation.
- Training is not performed in this notebook. Member 4 owns optimizer/hyperparameter experiments, callbacks, training, checkpoint selection, and training metrics.